# SalesPath — Colab Training Notebook

**Stack:** OpenEnv + GRPO (TRL) + Unsloth + Qwen 2.5

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Add `HF_TOKEN` in Colab Secrets (left sidebar 🔑)
3. Run **Cell 1** once (installs + clones)
4. Run **Cell 2** (starts server + validates)
5. Run **Cell 3** (curriculum training)
6. Run **Cell 4** (GRPO training)
7. Run **Cell 5** (reward graph)

In [1]:
# ============================================================
# CELL 1 — Install + Clone
# ============================================================
import os, sys, subprocess, time
from pathlib import Path

# ---------- CONFIG (edit these) ----------
REPO_URL          = "https://github.com/Imsachin010/salespath_env.git"
MODEL_NAME        = "Qwen/Qwen2.5-0.5B-Instruct"   # swap to 7B when VRAM allows
ENV_URL           = "http://127.0.0.1:8000"
CURRICULUM_STEPS  = 50
GRPO_STEPS        = 100
GRPO_DATASET_SIZE = 256
OUTPUT_DIR        = "/content/salespath_out"
PUSH_TO_HUB       = False
HUB_REPO          = "Imsachin010/salespath-qwen25-7b"
# -----------------------------------------

def run(cmd, check=True, cwd=None):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, cwd=cwd)
    if r.stdout: print(r.stdout.strip())
    if r.stderr: print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r

!nvidia-smi
print("Python:", sys.version)

# Install dependencies
!pip install -q -U pip
!pip uninstall -y openenv 2>/dev/null || true
!pip install -q fastapi uvicorn pydantic httpx openenv-core torch transformers trl unsloth datasets pyarrow huggingface_hub matplotlib

# Clone repo
if not Path("/content/salespath_env").exists():
    run(f"git clone {REPO_URL} /content/salespath_env")
else:
    print("Repo already cloned.")

# *** KEY FIX: always work from the REPO ROOT (where pyproject.toml lives) ***
REPO_ROOT = "/content/salespath_env"
os.chdir(REPO_ROOT)
print("Working dir:", os.getcwd())

# Install package in editable mode
run("pip install -q -e .")
run("python -c \"import salespath_env; print('salespath_env import OK')\"")
run("python -c \"import openenv.core; print('openenv.core import OK')\"")

# HF Login
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    from huggingface_hub import login
    login(token=hf_token)
    print("HF login OK")
else:
    print("HF_TOKEN not set — push steps will be skipped.")

print("\n✅ Setup complete.")

Sun Apr 26 04:28:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================================
# CELL 2 — Start Server + Validate
# ============================================================
import os, time, httpx
os.chdir("/content/salespath_env")

# Kill old server if any
!pkill -f 'uvicorn salespath_env.server.app' 2>/dev/null || true
time.sleep(1)

# Start server from REPO ROOT so module resolution works
!nohup python -m uvicorn salespath_env.server.app:app --host 0.0.0.0 --port 8000 > /content/server.log 2>&1 &
time.sleep(4)

# Health check with retries
healthy = False
for i in range(15):
    try:
        r = httpx.get("http://127.0.0.1:8000/health", timeout=5)
        if r.status_code == 200:
            print(f"✅ Server healthy: {r.text}")
            healthy = True
            break
    except Exception:
        pass
    time.sleep(2)
    print(f"  waiting... ({i+1}/15)")

if not healthy:
    print("\n--- server.log ---")
    !cat /content/server.log
    raise RuntimeError("Server failed to start! See logs above.")

# Quick API smoke test
reset_r = httpx.post("http://127.0.0.1:8000/reset", json={"difficulty": 1}, timeout=10)
print("\n/reset status:", reset_r.status_code)
obs = reset_r.json()
print("prospect_response:", obs.get("observation", obs).get("prospect_response", "")[:100])
print("\n✅ Server validation passed.")

^C
  waiting... (1/15)
✅ Server healthy: {"status":"healthy"}

/reset status: 200
prospect_response: You are engaging Meridian Retail, a medium retail company. Pain points: manual inventory tracking, s

✅ Server validation passed.


In [3]:
# ============================================================
# CELL 3 — Rollout Smoke Test (1 episode)
# ============================================================
import os
os.chdir("/content/salespath_env")
!python -m training.test_rollout

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/salespath_env/training/test_rollout.py", line 14, in <module>
    from rollout import run_episode
ModuleNotFoundError: No module named 'rollout'


In [4]:
# ============================================================
# CELL 4 — Curriculum Training (rollout loop, no grad update)
# Gets reward data + prints reward curve
# ============================================================
import os
os.chdir("/content/salespath_env")

!python -m training.grpo_train \
    --mode curriculum \
    --model-name Qwen/Qwen2.5-0.5B-Instruct \
    --env-url http://127.0.0.1:8000 \
    --steps 50 \
    --print-every 5 \
    --output-dir /content/salespath_out

Loading model: Qwen/Qwen2.5-0.5B-Instruct
tokenizer_config.json: 7.30kB [00:00, 5.25MB/s]
vocab.json: 2.78MB [00:00, 126MB/s]
merges.txt: 1.67MB [00:00, 125MB/s]
tokenizer.json: 7.03MB [00:00, 186MB/s]
config.json: 100% 659/659 [00:00<00:00, 6.25MB/s]
2026-04-26 04:21:27.919463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777177287.952378   10006 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777177287.966258   10006 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777177287.992639   10006 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:0

In [ ]:
# ============================================================
# CELL 5 — GRPO Training (gradient updates via TRL)
# ============================================================
import os
os.chdir("/content/salespath_env")

grpo_cmd = (
    "python -m training.grpo_train "
    "--mode grpo "
    "--model-name Qwen/Qwen2.5-0.5B-Instruct "
    "--grpo-steps 100 "
    "--grpo-dataset-size 256 "
    "--num-generations 4 "
    "--max-completion-length 64 "
    "--output-dir /content/salespath_out "
    "--logging-steps 5"
)
!{grpo_cmd}

In [ ]:
# ============================================================
# CELL 6 — Reward Graph
# ============================================================
import os
os.chdir("/content/salespath_env")

!python training/plot_rewards.py \
    --input /content/salespath_out/reward_history.txt \
    --output /content/salespath_out/reward_graph.png

# Display inline
from IPython.display import Image
Image("/content/salespath_out/reward_graph.png")

## Optional: Push Merged Model to HuggingFace

Set `HF_TOKEN` in Colab Secrets, then run:

```bash
python -m training.grpo_train \
    --mode curriculum \
    --steps 100 \
    --push-merged \
    --hub-repo Imsachin010/salespath-qwen25-7b
```